## the winning features

| Feature | Built from | Why this form |
|---|---|---|
| `log_vehicle_flight` | Vehicle flight number | RQ1: failure risk fell fastest over the first few flights, so a log scale fits the learning curve |
| `log_family_flight` | Family flight number | Tests whether experience carries over between variants of the same rocket family |
| `log_agency_flight` | Provider launch count | Captures the launch provider's overall experience |
| `vehicle_prior_failures` | Failures before this launch | The rocket's track record so far |
| `return_to_flight` | First flight after a failure | Tests whether launches right after a failure are riskier |
| `provider_group` | Provider class | RQ2: commercial vs. government |
| `decade` | Launch year | Failure rates dropped sharply from the 1950s to the 1970s, so era must be controlled |
| `n_stages` | Core stages, capped at 4 | Risk rose with each added stage; 5 and 6 stages were too rare to stand alone |
| `has_strapons` | Strap-on boosters (stage 0) | Design feature; boosters are common on mature heavy-lift rockets |
| `log_launch_mass` | Launch mass | Small rockets failed about 3× as often; log scale handles the huge range in mass |
| `orbit_group` | Target orbit, grouped into 5 categories | Deep space missions were riskiest and station missions safest |
| `country_group` | Provider country, with USSR and Russia combined and small programs as Other | Smaller, newer national programs failed more often |

| Excluded | Reason |
|---|---|
| `launch_class` (O vs. D) | Leaks the outcome: failed deep space launches are recorded as orbital |
| `launch_tag` | Leaks the outcome: failed launches receive an F designation |
| Stage disposal codes | Mostly reflect era and debris practices rather than risk |
| Length, diameter, thrust, LEO capacity | Overlap heavily with launch mass |
| Rocket name, provider name | Too many categories; experience and family capture this information |

In [0]:
# =============================================================================
# Modeling setup: data, splits, preprocessing, evaluation, MLflow logging
# =============================================================================

import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score,
                             precision_score, f1_score, accuracy_score, confusion_matrix)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Log only what we choose, so every run has the same clean fields
mlflow.sklearn.autolog(disable=True)


# -----------------------------------------------------------------------------
# 1. Load the modeling dataset into pandas (7,407 rows; small enough for memory)
# -----------------------------------------------------------------------------
pdf = spark.table("launch_reliability.gold.model_dataset").toPandas()

# Feature groups: each type gets different preprocessing
numeric_features = [
    "log_vehicle_flight", "log_family_flight", "log_agency_flight",
    "vehicle_prior_failures", "n_stages", "log_launch_mass",
]
binary_features = ["return_to_flight", "has_strapons"]
categorical_features = ["provider_group", "decade", "orbit_group", "country_group"]
features = numeric_features + binary_features + categorical_features

# True/False -> 1/0; decade as text so models treat each decade as a category
for c in binary_features:
    pdf[c] = pdf[c].astype(int)
pdf["decade"] = pdf["decade"].astype(str)


# -----------------------------------------------------------------------------
# 2. Time-based splits: learn from the past, evaluate on the future
#    fit (1957-2004)  -> train models while comparing and tuning
#    val (2005-2014)  -> choose models, settings, and thresholds
#    train (1957-2014) = fit + val, for the final refit
#    test (2015-2026) -> final evaluation, used ONCE at the end
# -----------------------------------------------------------------------------
pdf["year"] = pd.to_datetime(pdf["launch_ts"]).dt.year

fit   = pdf[pdf["year"] <= 2004]
val   = pdf[(pdf["year"] >= 2005) & (pdf["year"] <= 2014)]
train = pdf[pdf["year"] <= 2014]
test  = pdf[pdf["year"] >= 2015]

def xy(df):
    """Split a dataframe into features (X) and the failure label (y)."""
    return df[features], df["failed"].astype(int)

X_fit,   y_fit   = xy(fit)
X_val,   y_val   = xy(val)
X_train, y_train = xy(train)
X_test,  y_test  = xy(test)

for name, y in [("fit", y_fit), ("val", y_val), ("train", y_train), ("test", y_test)]:
    print(f"{name:6s} {len(y):5,} rows  {y.sum():4} failures  ({y.mean():.1%})")


# -----------------------------------------------------------------------------
# 3. Preprocessing (fit on training data only, then applied to evaluation data)
#    numbers:  fill missing with median, then standardize (needed for KNN, SVM, lasso)
#    binary:   pass through unchanged
#    category: one-hot encode; drop one level as the baseline;
#              ignore categories unseen in training (e.g., a new decade)
# -----------------------------------------------------------------------------
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("bin", "passthrough", binary_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features),
])


# -----------------------------------------------------------------------------
# 4. Evaluation: turn predicted probabilities into metrics
#    recall    = share of real failures the model caught
#    precision = share of flagged launches that really failed
#    roc_auc   = how well the model ranks risky launches above safe ones
#    pr_auc    = ranking skill focused on rare failures (random = failure rate)
#    tp/fp/fn/tn = caught failures / false alarms / missed failures / correct successes
# -----------------------------------------------------------------------------
def evaluate(model, X, y, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]          # predicted probability of failure
    pred = (proba >= threshold).astype(int)        # flag as failure above the threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "roc_auc":   roc_auc_score(y, proba),
        "pr_auc":    average_precision_score(y, proba),
        "recall":    recall_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "f1":        f1_score(y, pred, zero_division=0),
        "accuracy":  accuracy_score(y, pred),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


# -----------------------------------------------------------------------------
# 5. Fit, evaluate, and log one model to MLflow
#    stage="validation" (default): fit on 1957-2004, score on 2005-2014
#    stage="final":                fit on 1957-2014, score on 2015-2026 (once, at the end)
# -----------------------------------------------------------------------------
def run_model(name, model, imbalance="none", threshold=0.5, params=None, stage="validation"):
    if stage == "validation":
        X_tr, y_tr, X_ev, y_ev = X_fit, y_fit, X_val, y_val
        years = ("1957-2004", "2005-2014")
    else:
        X_tr, y_tr, X_ev, y_ev = X_train, y_train, X_test, y_test
        years = ("1957-2014", "2015-2026")

    # Fresh copy of preprocessing for each model, so no fitted state is shared
    pipe = Pipeline([("prep", clone(preprocess)), ("model", model)])

    with mlflow.start_run(run_name=f"{name} [{stage}]"):
        pipe.fit(X_tr, y_tr)
        metrics = evaluate(pipe, X_ev, y_ev, threshold)

        # Settings (params) and scores (metrics) show up as columns in MLflow
        mlflow.log_params({
            "model": name, "imbalance": imbalance, "threshold": threshold,
            "label": "failed", "stage": stage,
            "train_years": years[0], "eval_years": years[1],
            **(params or {}),
        })
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipe, "model", input_example=X_tr.head(5))

    return pipe, metrics


# -----------------------------------------------------------------------------
# 6. First validation runs: logistic regression, plain vs. class-weighted
#    class_weight="balanced" makes each missed failure count more in training
# -----------------------------------------------------------------------------
_, m_plain = run_model("logistic", LogisticRegression(max_iter=1000))

_, m_weighted = run_model(
    "logistic",
    LogisticRegression(max_iter=1000, class_weight="balanced"),
    imbalance="class_weight",
)

display(pd.DataFrame([m_plain, m_weighted], index=["plain", "class_weight"]).round(3))

In [0]:
# =============================================================================
# Model comparison on validation (fit 1957-2004, score 2005-2014)
# Compare on ROC AUC and PR AUC: they measure ranking skill and don't depend
# on the 0.5 threshold, so models can be compared fairly before threshold tuning.
# =============================================================================

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# LDA, QDA, and naive Bayes need a dense (non-sparse) matrix after one-hot encoding
preprocess.set_params(sparse_threshold=0)

balanced_priors = [0.5, 0.5]   # treat both classes as equally likely (balancing for LDA/QDA/NB)

models = {
    # Logistic regression with an L1 (lasso) penalty: shrinks weak features to exactly 0
    "lasso_logistic": (LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                                          class_weight="balanced", max_iter=2000),
                       "class_weight", {"C": 0.1}),

    # LDA: assumes each class is a normal distribution with a shared covariance
    "lda": (LinearDiscriminantAnalysis(priors=balanced_priors),
            "balanced_priors", {}),

    # QDA: like LDA, but each class gets its own covariance; reg_param keeps it stable
    "qda": (QuadraticDiscriminantAnalysis(priors=balanced_priors, reg_param=0.1),
            "balanced_priors", {"reg_param": 0.1}),

    # Naive Bayes: assumes features are independent given the class
    "naive_bayes": (GaussianNB(priors=balanced_priors),
                    "balanced_priors", {}),

    # KNN: predicts from the 25 most similar past launches (no class weight option)
    "knn": (KNeighborsClassifier(n_neighbors=25, weights="distance"),
            "none", {"n_neighbors": 25}),

    # SVM: finds a boundary with the widest margin; probability=True for AUC
    "svm": (SVC(kernel="rbf", C=1.0, probability=True, class_weight="balanced", random_state=42),
            "class_weight", {"C": 1.0, "kernel": "rbf"}),

    # Random forest: averages many decision trees, each on a bootstrap sample
    "random_forest": (RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
                                             class_weight="balanced_subsample",
                                             random_state=42, n_jobs=-1),
                      "class_weight", {"n_estimators": 500, "min_samples_leaf": 5}),

    # Gradient boosting: builds trees one at a time, each fixing the last one's mistakes
    "gradient_boosting": (HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                         max_iter=300, class_weight="balanced",
                                                         random_state=42),
                          "class_weight", {"max_depth": 3, "learning_rate": 0.05, "max_iter": 300}),
}

results = {"logistic (plain)": m_plain, "logistic": m_weighted}

for name, (model, imbalance, params) in models.items():
    print(f"running {name}...")
    _, results[name] = run_model(name, model, imbalance=imbalance, params=params)

comparison = (pd.DataFrame(results).T
              [["roc_auc", "pr_auc", "recall", "precision", "tp", "fp", "fn"]]
              .sort_values("pr_auc", ascending=False)
              .round(3))
display(comparison)

- Top tier: gradient boosting, QDA, random forest (PR AUC ~0.26–0.28)
- Middle tier: KNN, lasso, logistic (~0.18–0.23)
- Bottom tier: LDA, naive Bayes, SVM (~0.12–0.14)



- Ten models compared on 2005–2014 validation, all trained on 1957–2004
- Gradient boosting (PR AUC 0.277), QDA (0.269), and random forest (0.259) performed best
- All ROC AUCs fell between 0.66 and 0.72; with only 39 validation failures, differences of a few points may be noise
- The best models scored about 5× better than random on PR AUC, showing real but limited predictability
- SVM caught no failures at the 0.5 threshold because its probability calibration ignores class weights

In [0]:
# =============================================================================
# Tune the top models on validation (fit 1957-2004, score 2005-2014)
# Each combination is logged to MLflow; the best per model is chosen by PR AUC.
# Grids are kept small on purpose: with 39 validation failures, fine-grained
# tuning would mostly fit noise.
# =============================================================================

from itertools import product

def grid(**options):
    """Turn {'a': [1, 2], 'b': [3]} into [{'a': 1, 'b': 3}, {'a': 2, 'b': 3}]."""
    keys = list(options)
    return [dict(zip(keys, combo)) for combo in product(*options.values())]

tuning = {
    # Gradient boosting: tree depth, step size, and number of trees
    "gradient_boosting": (
        lambda p: HistGradientBoostingClassifier(**p, class_weight="balanced", random_state=42),
        "class_weight",
        grid(max_depth=[2, 3, 4], learning_rate=[0.03, 0.1], max_iter=[200, 500]),
    ),

    # Random forest: minimum leaf size (smoothness) and features tried per split
    "random_forest": (
        lambda p: RandomForestClassifier(**p, n_estimators=500, class_weight="balanced_subsample",
                                         random_state=42, n_jobs=-1),
        "class_weight",
        grid(min_samples_leaf=[1, 5, 10, 20], max_features=["sqrt", 0.5]),
    ),

    # Lasso: C is the inverse of penalty strength (smaller C = more features dropped)
    "lasso_logistic": (
        lambda p: LogisticRegression(**p, penalty="l1", solver="liblinear",
                                     class_weight="balanced", max_iter=2000),
        "class_weight",
        grid(C=[0.01, 0.03, 0.1, 0.3, 1.0]),
    ),

    # QDA: regularization blends each class's covariance toward simpler estimates
    "qda": (
        lambda p: QuadraticDiscriminantAnalysis(**p, priors=balanced_priors),
        "balanced_priors",
        grid(reg_param=[0.01, 0.1, 0.3, 0.5]),
    ),
}

best = {}
for name, (make_model, imbalance, param_grid) in tuning.items():
    for params in param_grid:
        print(f"{name} {params}")
        _, metrics = run_model(name, make_model(params), imbalance=imbalance, params=params)
        if name not in best or metrics["pr_auc"] > best[name]["metrics"]["pr_auc"]:
            best[name] = {"params": params, "metrics": metrics}

display(pd.DataFrame([
    {"model": name, **b["params"],
     "pr_auc": round(b["metrics"]["pr_auc"], 3),
     "roc_auc": round(b["metrics"]["roc_auc"], 3)}
    for name, b in best.items()
]).sort_values("pr_auc", ascending=False))

- Tuned 4 models across 29 settings on 2005–2014 validation
- Best: gradient boosting (PR AUC 0.294), random forest (0.272), QDA (0.269), lasso (0.215)
- Tuning improved PR AUC by at most ~0.02 and didn’t change the ranking, suggesting performance is limited by the features rather than model settings
- Slower learning (boosting) and larger leaves (random forest) worked best, consistent with avoiding overfitting on rare failures

In [0]:
# =============================================================================
# Threshold tuning on validation (fit 1957-2004, score 2005-2014)
# For each tuned model, choose the threshold that maximizes F1: the best
# balance between catching failures (recall) and avoiding false alarms (precision).
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Final settings chosen from tuning
chosen = {
    "gradient_boosting": (
        HistGradientBoostingClassifier(max_depth=4, learning_rate=0.03, max_iter=200,
                                       class_weight="balanced", random_state=42),
        "class_weight", {"max_depth": 4, "learning_rate": 0.03, "max_iter": 200}),
    "random_forest": (
        RandomForestClassifier(n_estimators=500, min_samples_leaf=20, max_features=0.5,
                               class_weight="balanced_subsample", random_state=42, n_jobs=-1),
        "class_weight", {"n_estimators": 500, "min_samples_leaf": 20, "max_features": 0.5}),
    "qda": (
        QuadraticDiscriminantAnalysis(reg_param=0.1, priors=balanced_priors),
        "balanced_priors", {"reg_param": 0.1}),
    "lasso_logistic": (
        LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                           class_weight="balanced", max_iter=2000),
        "class_weight", {"C": 0.1}),
}

thresholds = {}
fig, ax = plt.subplots(figsize=(7, 5))

for name, (model, imbalance, params) in chosen.items():
    # Fit on 1957-2004, get failure probabilities for 2005-2014
    pipe = Pipeline([("prep", clone(preprocess)), ("model", clone(model))])
    pipe.fit(X_fit, y_fit)
    proba = pipe.predict_proba(X_val)[:, 1]

    # Precision and recall at every possible threshold
    precision, recall, cutoffs = precision_recall_curve(y_val, proba)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_i = np.argmax(f1)
    thresholds[name] = float(cutoffs[best_i])

    print(f"{name:18s} threshold={cutoffs[best_i]:.3f}  "
          f"recall={recall[best_i]:.2f}  precision={precision[best_i]:.2f}  f1={f1[best_i]:.3f}")

    # Curve plus a dot at the chosen threshold
    line, = ax.plot(recall, precision, label=name)
    ax.scatter(recall[best_i], precision[best_i], color=line.get_color(), zorder=3)

# Random-guessing baseline: precision equals the failure rate at every recall
ax.axhline(y_val.mean(), color="gray", linestyle="--", label=f"random ({y_val.mean():.3f})")
ax.set_xlabel("Recall (share of failures caught)")
ax.set_ylabel("Precision (share of flagged launches that failed)")
ax.set_title("Precision-recall curves on validation (2005–2014)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# Log each model at its tuned threshold to MLflow
tuned = {}
for name, (model, imbalance, params) in chosen.items():
    _, tuned[name] = run_model(name, clone(model), imbalance=imbalance,
                               threshold=thresholds[name], params=params)

display(pd.DataFrame(tuned).T[["recall", "precision", "f1", "tp", "fp", "fn", "pr_auc", "roc_auc"]]
        .assign(threshold=pd.Series(thresholds))
        .round(3)
        .sort_values("f1", ascending=False))

In [0]:
# =============================================================================
# FINAL EVALUATION: fit on 1957-2014, score on 2015-2026 (run once)
# Model settings and thresholds were chosen on validation; nothing here is tuned.
# =============================================================================

final_results, final_models = {}, {}
fig, ax = plt.subplots(figsize=(7, 5))

for name, (model, imbalance, params) in chosen.items():
    pipe, metrics = run_model(name, clone(model), imbalance=imbalance,
                              threshold=thresholds[name], params=params, stage="final")
    final_models[name], final_results[name] = pipe, metrics

    # Precision-recall curve on the test set, with a dot at the tuned threshold
    proba = pipe.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, proba)
    line, = ax.plot(recall, precision, label=name)
    ax.scatter(metrics["recall"], metrics["precision"], color=line.get_color(), zorder=3)

ax.axhline(y_test.mean(), color="gray", linestyle="--", label=f"random ({y_test.mean():.3f})")
ax.set_xlabel("Recall (share of failures caught)")
ax.set_ylabel("Precision (share of flagged launches that failed)")
ax.set_title("Precision-recall curves on test (2015–2026)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

display(pd.DataFrame(final_results).T
        [["roc_auc", "pr_auc", "recall", "precision", "f1", "tp", "fp", "fn"]]
        .assign(threshold=pd.Series(thresholds))
        .round(3)
        .sort_values("pr_auc", ascending=False))

RQ4: How well can failure be predicted from pre-launch information?
- Four tuned models were refit on 1957–2014 and evaluated once on 2015–2026 (1,981 launches, 82 failures)
- Best PR AUC was 0.358 (gradient boosting), about 9× random; ROC AUCs ranged 0.81–0.87
At tuned thresholds, models caught 27–39% of failures, and 38–52% of flagged launches failed, 10–13× the base rate
- Lasso logistic regression performed as well as tree ensembles, with the highest ROC AUC (0.869) and the most failures caught (32)
- Models reliably identify a small set of high-risk launches, but most failures occur on launches that look routine before liftoff

In [0]:
# =============================================================================
# RQ3, part 1: which features does lasso keep, and in which direction?
# Positive coefficient = higher failure risk; zero = dropped by the penalty.
# =============================================================================

lasso = final_models["lasso_logistic"]
names = lasso.named_steps["prep"].get_feature_names_out()
coefs = lasso.named_steps["model"].coef_[0]

lasso_table = (pd.DataFrame({"feature": names, "coefficient": coefs})
               .assign(odds_ratio=lambda d: np.exp(d["coefficient"]))
               .query("coefficient != 0")
               .sort_values("coefficient", key=abs, ascending=False)
               .round(3))

print(f"Lasso kept {len(lasso_table)} of {len(names)} features")
display(lasso_table)

In [0]:
# =============================================================================
# RQ3, part 2: permutation importance on the test set
# Shuffle one feature at a time and measure how much PR AUC drops.
# A big drop = the model relies on that feature.
# =============================================================================

from sklearn.inspection import permutation_importance

importance = {}
for name, pipe in final_models.items():
    result = permutation_importance(pipe, X_test, y_test, scoring="average_precision",
                                    n_repeats=20, random_state=42, n_jobs=-1)
    importance[name] = pd.Series(result.importances_mean, index=X_test.columns)

importance_table = pd.DataFrame(importance).round(4)
importance_table["average"] = importance_table.mean(axis=1)
importance_table = importance_table.sort_values("average", ascending=False)
display(importance_table)

# Chart: average importance across the four models
fig, ax = plt.subplots(figsize=(7, 5))
importance_table["average"].sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("Drop in PR AUC when feature is shuffled")
ax.set_title("Which features matter most for predicting failure")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

RQ3: Which design and mission factors carry the most risk?
- Experience dominates: family, vehicle, and provider flight numbers were three of the four most important features across all four models
- Family experience was the single strongest predictor, so reliability carries over to new variants of established rocket families
- Launch mass was the only design feature with substantial importance; bigger rockets were slightly safer
- Stages, strap-on boosters, and target orbit added little once experience and size were included; lasso dropped strap-ons entirely
Historically, return-to-flight launches (odds ratio 1.56) and government providers (1.47) carried higher risk, but these patterns didn’t improve predictions for 2015–2026, suggesting they weakened over time